# Experiment 2 — Freeze U28 and learn the rest of CODI's global LM head

## Question

The bridge experiment showed that CODI's learned rank-96 head captures 58.6% of the
original U28 answer-initiation space. Retaining U28 at the first token preserved
91.7% of rank-96 sequence accuracy, while removing it reduced accuracy from 42.68%
to 8.49%. But the fixed U28 head failed completely at later answer positions.

This experiment builds the combined architecture suggested by those results:

```text
rank 96 basis = [ frozen U28 | 68 learned trajectory directions ]
rank 64 prefix = [ frozen U28 | first 36 learned trajectory directions ]
```

The 28 columns never move. The residual columns are trained in U28's orthogonal
complement. At deployment the QR training parameterization is materialized into the
same two matrix multiplications as any ordinary rank-64 or rank-96 head—there is no
extra online projection.

## Fair controls

- **Unconstrained continuation:** the validated global head receives exactly the
  same extra objective, clean data, on-policy data count, and epochs.
- **Random-28 constraint:** an energy-matched, U28-orthogonal random rank-28 basis is
  frozen and receives the same learned residual and training budget.
- **Frozen original:** the already validated global ranks 64 and 96 show whether any
  apparent gain merely comes from continued training.

Rank 96 is the primary quality comparison. Rank 64 is a locked secondary test of the
useful engineering hypothesis: explicit mechanistic structure may allow the same
quality with fewer online MACs.

## 1. Configuration and immutable checkout

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "6a8d2e6"  # Last published dependency commit; new utility is embedded below.
REPO_DIR = "/kaggle/working/latent-reasoning"

REPRODUCTION_SUMMARY_INPUT = ""
COLON_STATES_INPUT = ""
READOUT_INPUT = ""
GLOBAL_HEAD_ARTIFACT_INPUT = ""
OUTPUT_ROOT = "/kaggle/working/codi_constrained_u28_global_head"

SEED = 20260907
FIT_QUESTIONS = 1024
SELECT_QUESTIONS = 256
ONPOLICY_QUESTIONS = 256
MAX_FIT_STATES = 4096
MAX_SELECT_STATES = 1024
MAX_ONPOLICY_STATES = 2048
RANKS = (64, 96)
FIXED_RANK = 28
CLEAN_EPOCHS = 4
ONPOLICY_EPOCHS = 2
DISTILL_BATCH_SIZE = 16
LEARNING_RATE = 2e-4
TEMPERATURE = 2.0
KL_WEIGHT = 1.0
TOKEN_WEIGHT = 0.25
MARGIN_WEIGHT = 0.25
NESTED_WEIGHT = 0.5
MINIMUM_MARGIN = 0.25
ANCHOR_STRENGTH = 1e-5
GENERATION_BATCH_SIZE = 32
MAX_NEW_TOKENS = 64
BOOTSTRAP_SAMPLES = 5000
TEST_LIMIT = 0

import copy, glob, json, os, pathlib, random, subprocess, sys, time
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", RUN_COMMIT], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
CODE_COMMIT = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True,
    text=True, check=True,
).stdout.strip()
assert CODE_COMMIT.startswith(RUN_COMMIT)
pathlib.Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
print("commit:", CODE_COMMIT)

## 2. Reproduce the validated environment

In [ ]:
PINNED_PACKAGES = {
    "transformers": "4.52.4", "peft": "0.15.2", "datasets": "3.6.0",
    "huggingface_hub": "0.32.4",
}
from importlib.metadata import PackageNotFoundError, version as package_version

def installed_package_version(name):
    try: return package_version(name)
    except PackageNotFoundError: return None

missing = [f"{name}=={wanted}" for name, wanted in PINNED_PACKAGES.items()
           if installed_package_version(name) != wanted]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
probe = (
    "from peft.import_utils import is_torchao_available\n"
    "try:\n    print('ok' if is_torchao_available() else 'absent')\n"
    "except ImportError as error:\n    print('incompatible:' + str(error))\n"
)
state = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
state = (state.stdout + state.stderr).strip(); print("torchao:", state)
if state.startswith("incompatible"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([
    sys.executable, "-m", "pytest", "-q", "tests/test_global_low_rank_head.py",
    "tests/test_subspace_bridge.py", "tests/test_official_codi_fast.py",
], check=True)

## 3. Inputs

Attach the same three completed datasets used by the bridge experiment:

1. corrected official CODI reproduction;
2. `codi-answer-colon-margin-geometry`;
3. `trajectory-whitened-global-low-rank-lm-head`.

The bridge output itself is not required: this notebook reconstructs U28 from the
frozen calibration covariance and validates the global-head artifact directly.

In [ ]:
def discover(explicit, pattern):
    if explicit:
        path = pathlib.Path(explicit); assert path.is_file(), path; return path
    matches = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert matches, f"Attach a Kaggle dataset containing {pattern}"
    return pathlib.Path(sorted(matches, key=lambda value: (len(value.split('/')), value))[0])

REPRODUCTION_SUMMARY = discover(
    REPRODUCTION_SUMMARY_INPUT,
    "official_codi_gpt2/eval/revision_fd641b3d/full_gsm8k/summary.json",
)
COLON_STATES = discover(COLON_STATES_INPUT, "colon_states_seed89/colon_states.pt")
READOUT = discover(READOUT_INPUT, "colon_states_seed89/readout.pt")
GLOBAL_HEAD_ARTIFACT = discover(GLOBAL_HEAD_ARTIFACT_INPUT, "global_low_rank_head.pt")
GLOBAL_HEAD_SUMMARY = GLOBAL_HEAD_ARTIFACT.with_name("summary.json")
assert GLOBAL_HEAD_SUMMARY.is_file()
print({"reproduction": str(REPRODUCTION_SUMMARY), "states": str(COLON_STATES),
       "readout": str(READOUT), "global_head": str(GLOBAL_HEAD_ARTIFACT)})

## 4. Load U28, one matched-random control, and the frozen global head

In [ ]:
import torch
from scripts.run_official_codi_endpoint_margin_sweep import load_margin_cache
from src.mech.endpoint_correctness_geometry import readout_matrix
from src.mech.endpoint_margin_geometry import (
    ANALYTIC_STATE, build_band_subspace, build_matched_random_subspaces,
    state_covariance,
)
from src.mech.global_low_rank_head import NestedLowRankVocabularyHead
from src.mech.subspace_bridge import orthonormal_row_space, subspace_overlap

torch.manual_seed(SEED); random.seed(SEED)
cache, readout_payload = load_margin_cache(COLON_STATES, READOUT)
assert cache["metadata"]["precision"] == "float32"
assert cache["parity_gate"]["passed"] and cache["full_parity_gate"]["passed"]
state_index = list(cache["state_order"]).index(ANALYTIC_STATE)
colon_calibration = cache["calibration_states"][:, state_index, :].float()
colon_centre = cache["student_mean"][ANALYTIC_STATE].float()
covariance = state_covariance(colon_calibration - colon_centre)
u28_subspace = build_band_subspace(
    covariance=covariance, start=4, stop=32, state=ANALYTIC_STATE,
)
random28_subspace = build_matched_random_subspaces(
    selected=u28_subspace, covariance=covariance, replicates=1, seed=SEED + 17,
)[0]
assert random28_subspace.target_attainable and random28_subspace.selected_orthogonal

readout_cpu = readout_matrix(readout_payload).float()
global_artifact = torch.load(GLOBAL_HEAD_ARTIFACT, map_location="cpu", weights_only=False)
global_summary = json.loads(GLOBAL_HEAD_SUMMARY.read_text(encoding="utf-8"))
assert global_artifact["contract"] == "trajectory_whitened_margin_distilled_global_lm_head_v1"
assert global_summary["experiment"] == global_artifact["contract"]
assert tuple(int(value) for value in global_artifact["ranks"]) == (32, 64, 96)
prior_full = global_summary["generation"]["full"]["numeric_exact_match"]
prior_r96 = global_summary["generation"]["whitened_margin_onpolicy_r96"]["numeric_exact_match"]
assert prior_r96 / prior_full >= 0.98

global_cpu = NestedLowRankVocabularyHead(768, readout_cpu.shape[0], (32, 64, 96))
global_cpu.load_state_dict(global_artifact["state_dict"]); global_cpu.disable_adaptive()
q96 = orthonormal_row_space(global_cpu.down.weight[:96].detach())
print({"U28": list(u28_subspace.basis.shape), "random28": list(random28_subspace.basis.shape),
       "prior_rank96_retention": prior_r96 / prior_full,
       "q96_capture_of_U28": subspace_overlap(u28_subspace.basis, q96).reference_capture_fraction})

## 5. Load the constrained-head implementation

In [ ]:
EMBEDDED_CONSTRAINED_SOURCE_SHA256 = 'b578388a6fd979558828f9a7d479ad4fdc5f70ccfee2d76839dae5514d291fd7'
print('constrained source sha256:', EMBEDDED_CONSTRAINED_SOURCE_SHA256)
try:
    from src.mech.constrained_global_head import (
        ConstrainedNestedVocabularyHead, distil_ranked_head,
        evaluate_ranked_head, residual_completion,
    )
except ModuleNotFoundError:
    # Self-contained fallback until this notebook's implementation commit is published.
    exec('"""Nested low-rank vocabulary heads with a frozen hidden-state subspace.\n\nThe first columns of the bottleneck basis are fixed (for example CODI\'s U28 answer\nsubspace).  Remaining columns are learned but projected into the orthogonal\ncomplement.  After training, every configured prefix materializes as an ordinary\ntwo-linear-layer vocabulary head; the QR parameterization is training-only.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nimport math\nfrom typing import Sequence\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom src.mech.eigenspace_readout import LowRankVocabularyHead\n\n\ndef _orthonormal_columns(values: torch.Tensor, *, name: str) -> torch.Tensor:\n    if values.ndim != 2 or min(values.shape) <= 0:\n        raise ValueError(f"{name} must be a non-empty matrix")\n    if values.shape[1] > values.shape[0]:\n        raise ValueError(f"{name} cannot exceed its ambient dimension")\n    if not torch.isfinite(values).all():\n        raise ValueError(f"{name} contains non-finite values")\n    columns = torch.linalg.qr(values.double(), mode="reduced").Q\n    return columns.to(device=values.device, dtype=values.dtype)\n\n\ndef residual_completion(\n    fixed_basis: torch.Tensor,\n    candidate_basis: torch.Tensor,\n    residual_rank: int,\n) -> torch.Tensor:\n    """Top candidate-space directions orthogonal to ``fixed_basis``.\n\n    The left singular vectors of ``(I-UU^T)Q`` preserve as much of the ordered\n    candidate subspace as possible while making the constraint exact.\n    """\n    fixed = _orthonormal_columns(fixed_basis, name="fixed_basis").double()\n    candidate = _orthonormal_columns(candidate_basis, name="candidate_basis").double()\n    if fixed.shape[0] != candidate.shape[0]:\n        raise ValueError("fixed and candidate bases must share an ambient space")\n    residual_rank = int(residual_rank)\n    available = fixed.shape[0] - fixed.shape[1]\n    if not 0 < residual_rank <= min(available, candidate.shape[1]):\n        raise ValueError("residual rank exceeds the available candidate complement")\n    projected = candidate - fixed @ (fixed.T @ candidate)\n    left, singular_values, _ = torch.linalg.svd(projected, full_matrices=False)\n    tolerance = (\n        max(projected.shape)\n        * torch.finfo(torch.float64).eps\n        * float(singular_values.max())\n    )\n    if int((singular_values > tolerance).sum()) < residual_rank:\n        raise ValueError("candidate basis has insufficient rank outside the fixed space")\n    return left[:, :residual_rank].to(\n        device=fixed_basis.device, dtype=fixed_basis.dtype\n    )\n\n\nclass ConstrainedNestedVocabularyHead(nn.Module):\n    """A nested vocabulary head whose first hidden directions never move."""\n\n    def __init__(\n        self,\n        fixed_basis: torch.Tensor,\n        residual_basis: torch.Tensor,\n        centre: torch.Tensor,\n        vocabulary_size: int,\n        ranks: Sequence[int],\n    ) -> None:\n        super().__init__()\n        fixed = _orthonormal_columns(fixed_basis.detach(), name="fixed_basis")\n        residual = residual_completion(\n            fixed, residual_basis.detach(), residual_basis.shape[1]\n        )\n        hidden_size, fixed_rank = fixed.shape\n        max_rank = fixed_rank + residual.shape[1]\n        checked = tuple(int(rank) for rank in ranks)\n        if tuple(sorted(set(checked))) != checked:\n            raise ValueError("ranks must be strictly increasing")\n        if not checked or checked[-1] != max_rank or any(\n            rank <= fixed_rank or rank > max_rank for rank in checked\n        ):\n            raise ValueError("ranks must end at fixed plus residual rank")\n        if centre.shape != (hidden_size,):\n            raise ValueError("centre width must match the hidden dimension")\n        if vocabulary_size <= 0:\n            raise ValueError("vocabulary_size must be positive")\n\n        self.hidden_size = int(hidden_size)\n        self.vocabulary_size = int(vocabulary_size)\n        self.fixed_rank = int(fixed_rank)\n        self.max_rank = int(max_rank)\n        self.ranks = checked\n        self.register_buffer("fixed_basis", fixed.clone())\n        self.register_buffer("centre", centre.detach().to(fixed).clone())\n        self.residual_raw = nn.Parameter(residual.clone())\n        self.up = nn.Linear(self.max_rank, self.vocabulary_size, bias=True)\n        self.active_rank = self.max_rank\n        self._answer_position: int | None = None\n\n    @classmethod\n    @torch.no_grad()\n    def from_projected_readout(\n        cls,\n        readout_weight: torch.Tensor,\n        fixed_basis: torch.Tensor,\n        residual_basis: torch.Tensor,\n        centre: torch.Tensor,\n        ranks: Sequence[int],\n        *,\n        readout_bias: torch.Tensor | None = None,\n    ) -> "ConstrainedNestedVocabularyHead":\n        if readout_weight.ndim != 2:\n            raise ValueError("readout weight must be [vocabulary, hidden]")\n        vocabulary_size, hidden_size = readout_weight.shape\n        if fixed_basis.shape[0] != hidden_size or residual_basis.shape[0] != hidden_size:\n            raise ValueError("basis and readout hidden dimensions do not match")\n        if readout_bias is not None and readout_bias.shape != (vocabulary_size,):\n            raise ValueError("readout bias must have one value per token")\n        module = cls(\n            fixed_basis, residual_basis, centre, vocabulary_size, ranks\n        ).to(device=readout_weight.device, dtype=readout_weight.dtype)\n        basis = module.current_basis().detach()\n        module.up.weight.copy_(readout_weight @ basis)\n        origin = centre.to(readout_weight)\n        bias = origin @ readout_weight.T\n        if readout_bias is not None:\n            bias = bias + readout_bias.to(readout_weight)\n        module.up.bias.copy_(bias)\n        return module\n\n    def set_rank(self, rank: int) -> None:\n        if int(rank) not in self.ranks:\n            raise ValueError(f"rank must be one of {self.ranks}")\n        self.active_rank = int(rank)\n\n    def set_answer_position(self, position: int | None) -> None:\n        self._answer_position = None if position is None else int(position)\n\n    def current_residual_basis(self) -> torch.Tensor:\n        fixed = self.fixed_basis\n        projected = self.residual_raw - fixed @ (fixed.T @ self.residual_raw)\n        return torch.linalg.qr(projected, mode="reduced").Q\n\n    def current_basis(self, rank: int | None = None) -> torch.Tensor:\n        chosen = self.max_rank if rank is None else int(rank)\n        if chosen not in self.ranks:\n            raise ValueError(f"rank must be one of {self.ranks}")\n        residual_count = chosen - self.fixed_rank\n        return torch.cat(\n            (self.fixed_basis, self.current_residual_basis()[:, :residual_count]),\n            dim=1,\n        )\n\n    def forward_rank(self, hidden_states: torch.Tensor, rank: int) -> torch.Tensor:\n        rank = int(rank)\n        basis = self.current_basis(rank)\n        coordinates = (hidden_states - self.centre) @ basis\n        return F.linear(coordinates, self.up.weight[:, :rank], self.up.bias)\n\n    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:\n        return self.forward_rank(hidden_states, self.active_rank)\n\n    @torch.no_grad()\n    def materialize(self, rank: int | None = None) -> LowRankVocabularyHead:\n        """Freeze one prefix into the ordinary deployment-time two-layer head."""\n        chosen = self.max_rank if rank is None else int(rank)\n        basis = self.current_basis(chosen).detach()\n        module = LowRankVocabularyHead(\n            self.hidden_size, self.vocabulary_size, chosen\n        ).to(device=self.up.weight.device, dtype=self.up.weight.dtype)\n        module.down.weight.copy_(basis.T)\n        module.down.bias.copy_(-(self.centre @ basis))\n        module.up.weight.copy_(self.up.weight[:, :chosen])\n        module.up.bias.copy_(self.up.bias)\n        return module\n\n\n@dataclass(frozen=True)\nclass RankedDistillationResult:\n    losses: tuple[float, ...]\n    validation: tuple[dict[str, float], ...]\n    best_epoch: int\n    best_top1_agreement: float\n    best_validation_kl: float\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\n@torch.no_grad()\ndef evaluate_ranked_head(\n    head: nn.Module,\n    states: torch.Tensor,\n    readout_weight: torch.Tensor,\n    *,\n    rank: int,\n    readout_bias: torch.Tensor | None = None,\n    batch_size: int = 16,\n    temperature: float = 2.0,\n) -> dict[str, float]:\n    device = next(head.parameters()).device\n    dtype = next(head.parameters()).dtype\n    weight = readout_weight.detach().to(device=device, dtype=dtype)\n    bias = None if readout_bias is None else readout_bias.detach().to(device=device, dtype=dtype)\n    total = agreements = 0\n    total_kl = margin_error = 0.0\n    head.eval()\n    for start in range(0, len(states), int(batch_size)):\n        hidden = states[start : start + int(batch_size)].to(device=device, dtype=dtype)\n        teacher = F.linear(hidden, weight, bias).float()\n        student = head.forward_rank(hidden, int(rank)).float()\n        probability = F.softmax(teacher / temperature, dim=-1)\n        divergence = F.kl_div(\n            F.log_softmax(student / temperature, dim=-1), probability,\n            reduction="batchmean",\n        )\n        teacher_margin = teacher.topk(2, dim=-1).values.diff(dim=-1).abs().squeeze(-1)\n        student_margin = student.topk(2, dim=-1).values.diff(dim=-1).abs().squeeze(-1)\n        count = len(hidden)\n        total += count\n        agreements += int((student.argmax(-1) == teacher.argmax(-1)).sum())\n        total_kl += float(divergence) * count\n        margin_error += float((student_margin - teacher_margin).abs().sum())\n    return {\n        "examples": int(total),\n        "kl": total_kl / max(1, total),\n        "top1_agreement": agreements / max(1, total),\n        "mean_absolute_top2_margin_error": margin_error / max(1, total),\n    }\n\n\ndef _margin_loss(logits: torch.Tensor, teacher_top: torch.Tensor, margin: float) -> torch.Tensor:\n    selected = logits.gather(1, teacher_top.unsqueeze(1)).squeeze(1)\n    top_two = logits.topk(k=2, dim=-1)\n    competitor = torch.where(\n        top_two.indices[:, 0] == teacher_top,\n        top_two.values[:, 1],\n        top_two.values[:, 0],\n    )\n    return F.relu(float(margin) - selected + competitor).mean()\n\n\ndef distil_ranked_head(\n    head: nn.Module,\n    train_states: torch.Tensor,\n    validation_states: torch.Tensor,\n    readout_weight: torch.Tensor,\n    *,\n    ranks: Sequence[int],\n    readout_bias: torch.Tensor | None = None,\n    epochs: int = 4,\n    batch_size: int = 16,\n    learning_rate: float = 2e-4,\n    temperature: float = 2.0,\n    kl_weight: float = 1.0,\n    token_weight: float = 0.25,\n    margin_weight: float = 0.25,\n    nested_weight: float = 0.5,\n    minimum_margin: float = 0.25,\n    anchor_strength: float = 1e-5,\n    seed: int = 0,\n) -> RankedDistillationResult:\n    """Give constrained and unconstrained nested heads the same training objective."""\n    checked = tuple(int(rank) for rank in ranks)\n    if checked != tuple(getattr(head, "ranks", ())):\n        raise ValueError("requested ranks must exactly match the head")\n    if train_states.ndim != 2 or validation_states.ndim != 2:\n        raise ValueError("training and validation states must be matrices")\n    hidden_size = int(getattr(head, "hidden_size"))\n    vocabulary_size = int(getattr(head, "vocabulary_size"))\n    if train_states.shape[1] != hidden_size or validation_states.shape[1] != hidden_size:\n        raise ValueError("state width does not match head")\n    if readout_weight.shape != (vocabulary_size, hidden_size):\n        raise ValueError("readout dimensions do not match head")\n\n    device = next(head.parameters()).device\n    dtype = next(head.parameters()).dtype\n    teacher_weight = readout_weight.detach().to(device=device, dtype=dtype)\n    teacher_bias = None if readout_bias is None else readout_bias.detach().to(device=device, dtype=dtype)\n    anchor_parameter = (\n        head.residual_raw if hasattr(head, "residual_raw") else head.down.weight\n    )\n    initial_anchor = anchor_parameter.detach().clone()\n    optimizer = torch.optim.AdamW(head.parameters(), lr=float(learning_rate), weight_decay=0.0)\n    generator = torch.Generator(device="cpu").manual_seed(int(seed))\n    losses: list[float] = []\n    validation: list[dict[str, float]] = []\n    best_key = (-math.inf, -math.inf)\n    best_epoch = -1\n    best_state = None\n\n    for epoch in range(int(epochs)):\n        order = torch.randperm(len(train_states), generator=generator)\n        head.train()\n        loss_sum = 0.0\n        seen = 0\n        for start in range(0, len(order), int(batch_size)):\n            index = order[start : start + int(batch_size)]\n            hidden = train_states[index].to(device=device, dtype=dtype)\n            with torch.no_grad():\n                teacher = F.linear(hidden, teacher_weight, teacher_bias).float()\n                probability = F.softmax(teacher / temperature, dim=-1)\n                teacher_top = teacher.argmax(-1)\n\n            def loss_at(rank: int) -> torch.Tensor:\n                student = head.forward_rank(hidden, rank).float()\n                divergence = F.kl_div(\n                    F.log_softmax(student / temperature, dim=-1), probability,\n                    reduction="batchmean",\n                ) * (temperature * temperature)\n                token = F.cross_entropy(student, teacher_top)\n                ranking = _margin_loss(student, teacher_top, minimum_margin)\n                return (float(kl_weight) * divergence + float(token_weight) * token\n                        + float(margin_weight) * ranking)\n\n            loss = loss_at(checked[-1])\n            if len(checked) > 1:\n                auxiliary = torch.stack([loss_at(rank) for rank in checked[:-1]]).mean()\n                loss = loss + float(nested_weight) * auxiliary\n            loss = loss + float(anchor_strength) * (\n                anchor_parameter - initial_anchor\n            ).square().mean()\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            optimizer.step()\n            count = len(hidden)\n            seen += count\n            loss_sum += float(loss.detach()) * count\n        losses.append(loss_sum / max(1, seen))\n        metrics = evaluate_ranked_head(\n            head, validation_states, teacher_weight, rank=checked[-1],\n            readout_bias=teacher_bias, batch_size=batch_size,\n            temperature=temperature,\n        )\n        validation.append(metrics)\n        key = (metrics["top1_agreement"], -metrics["kl"])\n        if key > best_key:\n            best_key = key\n            best_epoch = epoch\n            best_state = {\n                name: value.detach().cpu().clone()\n                for name, value in head.state_dict().items()\n            }\n    if best_state is None:\n        raise RuntimeError("distillation did not produce a checkpoint")\n    head.load_state_dict(best_state)\n    best = validation[best_epoch]\n    return RankedDistillationResult(\n        losses=tuple(losses), validation=tuple(validation), best_epoch=best_epoch,\n        best_top1_agreement=float(best["top1_agreement"]),\n        best_validation_kl=float(best["kl"]),\n    )\n', globals())

## 6. Load CODI and freeze question-disjoint populations

In [ ]:
from scripts.collect_official_codi_endpoint_tsvc import verify_full_reproduction_gate
from src.data.datasets import load_eval_set, load_train_set
from src.inference.official_codi_fast import (
    generate_official_codi_fast, prepare_official_codi_batches,
)
from src.models.official_codi import (
    build_official_codi_gpt2, download_official_checkpoint, generate_official_codi,
    load_official_checkpoint, resolve_torch_dtype,
)
from src.utils.config import load_config

assert torch.cuda.is_available(), "Select a Kaggle GPU"
cfg = load_config("configs/official_codi_gpt2.yaml")
verify_full_reproduction_gate(REPRODUCTION_SUMMARY, cfg)
device = torch.device("cuda"); dtype = resolve_torch_dtype("float32", device)
checkpoint_matches = sorted(glob.glob(f"/kaggle/input/**/{cfg.checkpoint.filename}", recursive=True))
checkpoint = pathlib.Path(checkpoint_matches[0]) if checkpoint_matches else download_official_checkpoint(
    repo_id=str(cfg.checkpoint.repo_id), revision=str(cfg.checkpoint.revision),
    filename=str(cfg.checkpoint.filename), expected_sha256=str(cfg.checkpoint.sha256),
    token=os.environ.get("HF_TOKEN") or None,
)
model, tokenizer = build_official_codi_gpt2(
    base_model=str(cfg.model.base_model), base_revision=str(cfg.model.base_revision),
    dtype=dtype, settings=cfg.model, token=os.environ.get("HF_TOKEN") or None,
)
load_report = load_official_checkpoint(model, checkpoint, expected_sha256=str(cfg.checkpoint.sha256))
assert str(cache["metadata"]["checkpoint_sha256"]) == load_report.checkpoint_sha256
assert global_summary["checkpoint_sha256"] == load_report.checkpoint_sha256
for parameter in model.parameters(): parameter.requires_grad_(False)
model.to(device=device, dtype=dtype).eval()
base_model = model.codi.get_base_model(); full_head = base_model.get_output_embeddings()
vocabulary_size = int(model.eot_id); hidden_size = int(model.config.hidden_size)
readout_weight = full_head.weight[:vocabulary_size].detach()
assert hidden_size == 768 and tuple(readout_weight.shape) == tuple(readout_cpu.shape)
assert torch.allclose(readout_weight.cpu(), readout_cpu, atol=0, rtol=0)

training_rows = load_train_set(load_config("configs/data.yaml"), trace_style="eq_only")
required = FIT_QUESTIONS + SELECT_QUESTIONS + ONPOLICY_QUESTIONS
order = torch.randperm(len(training_rows), generator=torch.Generator().manual_seed(SEED))
assert required <= len(order)
fit_i = order[:FIT_QUESTIONS]
select_i = order[FIT_QUESTIONS:FIT_QUESTIONS + SELECT_QUESTIONS]
onpolicy_i = order[FIT_QUESTIONS + SELECT_QUESTIONS:required]
assert set(fit_i.tolist()).isdisjoint(select_i.tolist())
assert set(fit_i.tolist()).isdisjoint(onpolicy_i.tolist())
assert set(select_i.tolist()).isdisjoint(onpolicy_i.tolist())
questions = lambda indices: [str(training_rows[int(i)]["question"]) for i in indices]
fit_questions, select_questions, onpolicy_questions = (
    questions(fit_i), questions(select_i), questions(onpolicy_i)
)
print({"fit": len(fit_questions), "select": len(select_questions),
       "onpolicy": len(onpolicy_questions), "test": "locked until section 12"})

## 7. Verify the body-only decoder and collect clean trajectory states

In [ ]:
parity_questions = select_questions[:16]
reference = generate_official_codi(
    model, tokenizer, parity_questions, latent_iterations=int(cfg.eval.latent_iterations),
    max_new_tokens=MAX_NEW_TOKENS, batch_size=16, device=device,
    answer_cue="The answer is:", force_answer_cue=True,
)
prepared_parity = prepare_official_codi_batches(
    tokenizer, parity_questions, batch_size=16, length_bucketed=False,
)
fast = generate_official_codi_fast(
    model, tokenizer, prepared_parity, latent_iterations=int(cfg.eval.latent_iterations),
    max_new_tokens=MAX_NEW_TOKENS, device=device, answer_cue="The answer is:",
)
assert tuple(reference) == fast.texts
print({"fastpath_parity_examples": 16, "exact": True})

def deterministic_cap(states, maximum, seed, exact=None):
    wanted = min(len(states), int(maximum)) if exact is None else int(exact)
    assert 0 < wanted <= len(states)
    if wanted == len(states): return states
    generator = torch.Generator().manual_seed(int(seed))
    return states[torch.randperm(len(states), generator=generator)[:wanted]]

@torch.inference_mode()
def collect_states(question_list, output_head, tag, maximum):
    captured = []
    def observer(hidden, active, answer_position):
        del answer_position
        if bool(active.any()): captured.append(hidden[active].detach().cpu().float())
    base_model.set_output_embeddings(output_head)
    try:
        prepared = prepare_official_codi_batches(
            tokenizer, question_list, batch_size=GENERATION_BATCH_SIZE,
            length_bucketed=False,
        )
        generated = generate_official_codi_fast(
            model, tokenizer, prepared, latent_iterations=int(cfg.eval.latent_iterations),
            max_new_tokens=MAX_NEW_TOKENS, device=device, answer_cue="The answer is:",
            answer_state_observer=observer,
        )
    finally:
        base_model.set_output_embeddings(full_head)
    values = torch.cat(captured); values = deterministic_cap(values, maximum, SEED + len(tag))
    print(tag, {"states": len(values), "questions": len(generated.texts)})
    return values

fit_states = collect_states(fit_questions, full_head, "clean_fit", MAX_FIT_STATES)
select_states = collect_states(select_questions, full_head, "clean_select", MAX_SELECT_STATES)
trajectory_centre = fit_states.double().mean(0).float()
print({"fit_states": len(fit_states), "select_states": len(select_states)})

## 8. Construct equal-rank initializations

The residual starts from the part of the validated global rank-96 row space that is
orthogonal to the frozen 28-dimensional basis. Both U28 and random controls use the
same construction. Their initial up matrices are the dense readout projected onto
the resulting 96-dimensional basis around the same trajectory mean.

The unconstrained control starts from the exact validated global factors. It is
reconfigured to the same nested `(64, 96)` objective as the constrained heads.

In [ ]:
u_residual = residual_completion(u28_subspace.basis, q96, 96 - FIXED_RANK)
random_residual = residual_completion(random28_subspace.basis, q96, 96 - FIXED_RANK)

u_head = ConstrainedNestedVocabularyHead.from_projected_readout(
    readout_weight, u28_subspace.basis.to(device), u_residual.to(device),
    trajectory_centre.to(device), RANKS,
).to(device).eval()
random_head = ConstrainedNestedVocabularyHead.from_projected_readout(
    readout_weight, random28_subspace.basis.to(device), random_residual.to(device),
    trajectory_centre.to(device), RANKS,
).to(device).eval()
continued_head = NestedLowRankVocabularyHead(hidden_size, vocabulary_size, RANKS)
continued_head.load_state_dict(global_artifact["state_dict"], strict=True)
continued_head.disable_adaptive(); continued_head.to(device=device, dtype=dtype).eval()

def cpu_snapshot(head, rank):
    if isinstance(head, ConstrainedNestedVocabularyHead):
        return head.materialize(rank).cpu().eval()
    clone = copy.deepcopy(head).cpu().eval(); clone.set_rank(rank); return clone

snapshots = {}
for rank in RANKS:
    snapshots[f"u28_initial_r{rank}"] = cpu_snapshot(u_head, rank)

def metrics_for(head):
    return {f"rank{rank}": evaluate_ranked_head(
        head, select_states, readout_weight, rank=rank,
        batch_size=DISTILL_BATCH_SIZE, temperature=TEMPERATURE,
    ) for rank in RANKS}

initial_metrics = {"u28": metrics_for(u_head), "random28": metrics_for(random_head),
                   "continued_unconstrained": metrics_for(continued_head)}
print(json.dumps(initial_metrics, indent=2))

## 9. Equal-budget clean-trajectory distillation

In [ ]:
def train_one(head, states, epochs, seed):
    return distil_ranked_head(
        head, states, select_states, readout_weight, ranks=RANKS,
        epochs=epochs, batch_size=DISTILL_BATCH_SIZE, learning_rate=LEARNING_RATE,
        temperature=TEMPERATURE, kl_weight=KL_WEIGHT, token_weight=TOKEN_WEIGHT,
        margin_weight=MARGIN_WEIGHT, nested_weight=NESTED_WEIGHT,
        minimum_margin=MINIMUM_MARGIN, anchor_strength=ANCHOR_STRENGTH, seed=seed,
    )

clean_results = {}
for name, head in (("u28", u_head), ("random28", random_head),
                   ("continued_unconstrained", continued_head)):
    clean_results[name] = train_one(head, fit_states, CLEAN_EPOCHS, SEED + 100)
    print(name, clean_results[name].to_dict())
    torch.cuda.empty_cache()

for rank in RANKS:
    snapshots[f"u28_clean_r{rank}"] = cpu_snapshot(u_head, rank)
clean_metrics = {"u28": metrics_for(u_head), "random28": metrics_for(random_head),
                 "continued_unconstrained": metrics_for(continued_head)}
print(json.dumps(clean_metrics, indent=2))

## 10. Equal-count on-policy recovery

Each clean rank-96 head generates the same 256 questions. We collect the states
caused by its own decisions, then deterministically cap all three pools to the
smallest observed count. Thus every arm gets the same number of recovery examples,
while the examples still represent that arm's own trajectory distribution.

In [ ]:
onpolicy_raw = {
    "u28": collect_states(onpolicy_questions, cpu_snapshot(u_head, 96).to(device),
                           "onpolicy_u28", MAX_ONPOLICY_STATES),
    "random28": collect_states(onpolicy_questions, cpu_snapshot(random_head, 96).to(device),
                               "onpolicy_random28", MAX_ONPOLICY_STATES),
    "continued_unconstrained": collect_states(
        onpolicy_questions, cpu_snapshot(continued_head, 96).to(device),
        "onpolicy_continued", MAX_ONPOLICY_STATES,
    ),
}
common_onpolicy_count = min(len(values) for values in onpolicy_raw.values())
onpolicy_states = {
    name: deterministic_cap(values, common_onpolicy_count, SEED + 300 + offset,
                            exact=common_onpolicy_count)
    for offset, (name, values) in enumerate(onpolicy_raw.items())
}
assert len({len(values) for values in onpolicy_states.values()}) == 1
print({"common_onpolicy_states": common_onpolicy_count})

onpolicy_results = {}
for offset, (name, head) in enumerate((
    ("u28", u_head), ("random28", random_head),
    ("continued_unconstrained", continued_head),
)):
    recovery = torch.cat((fit_states, onpolicy_states[name]), dim=0)
    onpolicy_results[name] = train_one(head, recovery, ONPOLICY_EPOCHS, SEED + 400)
    print(name, onpolicy_results[name].to_dict())
    torch.cuda.empty_cache()

final_metrics = {"u28": metrics_for(u_head), "random28": metrics_for(random_head),
                 "continued_unconstrained": metrics_for(continued_head)}
for rank in RANKS:
    snapshots[f"u28_onpolicy_r{rank}"] = cpu_snapshot(u_head, rank)
    snapshots[f"random28_onpolicy_r{rank}"] = cpu_snapshot(random_head, rank)
    snapshots[f"continued_onpolicy_r{rank}"] = cpu_snapshot(continued_head, rank)
print(json.dumps(final_metrics, indent=2))

## 11. Verify the constraint and deployment contract before test evaluation

In [ ]:
constraint_checks = {}
for name, head, fixed in (("u28", u_head, u28_subspace.basis.to(device)),
                          ("random28", random_head, random28_subspace.basis.to(device))):
    basis96 = head.current_basis(96).detach()
    residual = basis96[:, FIXED_RANK:]
    deployed = head.materialize(96)
    probe_states = select_states[:32].to(device)
    constraint_checks[name] = {
        "fixed_basis_capture": subspace_overlap(
            fixed.cpu(), deployed.down.weight[:FIXED_RANK].detach().T.cpu()
        ).reference_capture_fraction,
        "maximum_fixed_residual_dot": float((fixed.T @ residual).abs().max()),
        "maximum_basis_gram_error": float(
            (basis96.T @ basis96 - torch.eye(96, device=device)).abs().max()
        ),
        "materialization_max_logit_error": float(
            (head.forward_rank(probe_states, 96) - deployed(probe_states)).abs().max()
        ),
        "U28_capture_by_deployed_space": subspace_overlap(
            u28_subspace.basis, orthonormal_row_space(
                deployed.down.weight.detach().cpu()
            )
        ).reference_capture_fraction,
    }
assert constraint_checks["u28"]["fixed_basis_capture"] > 0.99999
assert constraint_checks["u28"]["maximum_fixed_residual_dot"] < 1e-4
assert constraint_checks["u28"]["materialization_max_logit_error"] < 1e-3
print(json.dumps(constraint_checks, indent=2))

## 12. Locked full 1,319-example GSM8K generation

In [ ]:
from src.data.answer_extract import answers_match

test_examples = load_eval_set("gsm8k", load_config(cfg.data_config).eval.gsm8k)
assert len(test_examples) == 1319
if TEST_LIMIT: test_examples = test_examples[:TEST_LIMIT]
test_questions = [str(row["question"]) for row in test_examples]
prepared_test = prepare_official_codi_batches(
    tokenizer, test_questions, batch_size=GENERATION_BATCH_SIZE,
    length_bucketed=False,
)
prepared_warmup = prepare_official_codi_batches(
    tokenizer, select_questions[:32], batch_size=GENERATION_BATCH_SIZE,
    length_bucketed=False,
)

frozen_global = NestedLowRankVocabularyHead(hidden_size, vocabulary_size, (32, 64, 96))
frozen_global.load_state_dict(global_artifact["state_dict"]); frozen_global.disable_adaptive()
for rank in RANKS:
    head = copy.deepcopy(frozen_global).cpu().eval(); head.set_rank(rank)
    snapshots[f"frozen_global_r{rank}"] = head

deployment_arms = {"dense_full": None, **snapshots}
assert len(deployment_arms) == 13

def write_jsonl(path, records):
    temporary = pathlib.Path(str(path) + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    temporary.replace(path)

generation_results = {}; correctness = {}
for name, cpu_head in deployment_arms.items():
    head = full_head if cpu_head is None else cpu_head.to(device=device, dtype=dtype).eval()
    base_model.set_output_embeddings(head)
    _ = generate_official_codi_fast(
        model, tokenizer, prepared_warmup, latent_iterations=int(cfg.eval.latent_iterations),
        max_new_tokens=MAX_NEW_TOKENS, device=device, answer_cue="The answer is:",
    )
    torch.cuda.synchronize(); started = time.perf_counter()
    generated = generate_official_codi_fast(
        model, tokenizer, prepared_test, latent_iterations=int(cfg.eval.latent_iterations),
        max_new_tokens=MAX_NEW_TOKENS, device=device, answer_cue="The answer is:",
    )
    torch.cuda.synchronize(); elapsed = time.perf_counter() - started
    flags = [bool(answers_match(text, row["gold"]))
             for text, row in zip(generated.texts, test_examples)]
    correctness[name] = torch.tensor(flags, dtype=torch.bool)
    records = [{"index": index, "question": str(row["question"]),
                "gold": str(row["gold"]), "generation": text,
                "token_ids": list(ids), "correct": bool(flag)}
               for index, (row, text, ids, flag) in enumerate(zip(
                   test_examples, generated.texts, generated.token_ids, flags))]
    write_jsonl(pathlib.Path(OUTPUT_ROOT) / f"{name}.jsonl", records)
    generation_results[name] = {
        "examples": len(flags), "correct": int(sum(flags)),
        "numeric_exact_match": float(sum(flags) / len(flags)),
        "elapsed_seconds_single_run_not_a_benchmark": elapsed,
        "generated_tokens": generated.generated_token_count,
    }
    print(name, generation_results[name])
    base_model.set_output_embeddings(full_head)
    if cpu_head is not None: cpu_head.cpu(); torch.cuda.empty_cache()

## 13. Paired inference, gates, artifact export, and plots

In [ ]:
def paired_bootstrap(reference, candidate, samples, seed):
    difference = candidate.float() - reference.float()
    generator = torch.Generator().manual_seed(seed); estimates = []
    for _ in range(0, samples, 250):
        count = min(250, samples - len(estimates))
        index = torch.randint(len(difference), (count, len(difference)), generator=generator)
        estimates.extend(difference[index].mean(1).tolist())
    values = torch.tensor(estimates)
    return {"delta_accuracy": float(difference.mean()),
            "ci95_low": float(torch.quantile(values, 0.025)),
            "ci95_high": float(torch.quantile(values, 0.975)),
            "reference_correct_to_candidate_wrong": int((reference & ~candidate).sum()),
            "reference_wrong_to_candidate_correct": int((~reference & candidate).sum())}

contrasts = {
    "u28_r96_vs_frozen_r96": ("frozen_global_r96", "u28_onpolicy_r96"),
    "u28_r96_vs_equal_budget_unconstrained_r96": ("continued_onpolicy_r96", "u28_onpolicy_r96"),
    "u28_r96_vs_random_constraint_r96": ("random28_onpolicy_r96", "u28_onpolicy_r96"),
    "u28_r64_vs_frozen_r64": ("frozen_global_r64", "u28_onpolicy_r64"),
    "u28_r64_vs_frozen_r96": ("frozen_global_r96", "u28_onpolicy_r64"),
}
paired = {name: paired_bootstrap(correctness[reference], correctness[candidate],
                                  BOOTSTRAP_SAMPLES, SEED + 700 + offset)
          for offset, (name, (reference, candidate)) in enumerate(contrasts.items())}

accuracy = lambda name: generation_results[name]["numeric_exact_match"]
dense_accuracy = accuracy("dense_full")
decisions = {
    "reproduces_prior_dense_and_rank96_within_0_5pp": bool(
        abs(accuracy("dense_full") - prior_full) <= 0.005 and
        abs(accuracy("frozen_global_r96") - prior_r96) <= 0.005
    ),
    "u28_rank96_retains_98pct_dense": bool(accuracy("u28_onpolicy_r96") >= 0.98 * dense_accuracy),
    "u28_rank96_beats_frozen_rank96": bool(accuracy("u28_onpolicy_r96") > accuracy("frozen_global_r96")),
    "u28_rank96_beats_equal_budget_unconstrained": bool(
        accuracy("u28_onpolicy_r96") > accuracy("continued_onpolicy_r96")
    ),
    "u28_rank96_beats_random_constraint": bool(
        accuracy("u28_onpolicy_r96") > accuracy("random28_onpolicy_r96")
    ),
    "u28_rank96_paired_ci_above_equal_budget_unconstrained": bool(
        paired["u28_r96_vs_equal_budget_unconstrained_r96"]["ci95_low"] > 0
    ),
    "u28_rank96_paired_ci_above_random_constraint": bool(
        paired["u28_r96_vs_random_constraint_r96"]["ci95_low"] > 0
    ),
    "u28_rank64_retains_98pct_of_frozen_rank96": bool(
        accuracy("u28_onpolicy_r64") >= 0.98 * accuracy("frozen_global_r96")
    ),
}
decisions["constrained_quality_benefit_supported"] = bool(
    decisions["reproduces_prior_dense_and_rank96_within_0_5pp"] and
    decisions["u28_rank96_retains_98pct_dense"] and
    decisions["u28_rank96_paired_ci_above_equal_budget_unconstrained"] and
    decisions["u28_rank96_paired_ci_above_random_constraint"]
)
decisions["rank_reduction_supported"] = decisions[
    "u28_rank64_retains_98pct_of_frozen_rank96"
]

operations = {
    "dense": hidden_size * vocabulary_size,
    "rank64": hidden_size * 64 + 64 * vocabulary_size,
    "rank96": hidden_size * 96 + 96 * vocabulary_size,
}
operations["rank64_head_mac_reduction_vs_dense"] = operations["dense"] / operations["rank64"]
operations["rank96_head_mac_reduction_vs_dense"] = operations["dense"] / operations["rank96"]

summary = {
    "experiment": "official_codi_frozen_u28_learned_residual_global_head_v1",
    "code_commit": CODE_COMMIT, "checkpoint_sha256": load_report.checkpoint_sha256,
    "constrained_source_sha256": EMBEDDED_CONSTRAINED_SOURCE_SHA256,
    "population": {"fit_questions": FIT_QUESTIONS, "select_questions": SELECT_QUESTIONS,
                   "onpolicy_questions": ONPOLICY_QUESTIONS,
                   "fit_states": len(fit_states), "select_states": len(select_states),
                   "common_onpolicy_states": common_onpolicy_count,
                   "test_questions": len(test_examples)},
    "architecture": {"fixed_basis": "colon_state12_PC4_31", "fixed_rank": FIXED_RANK,
                     "ranks": list(RANKS), "rank64_residual": 36,
                     "rank96_residual": 68, "deployment_extra_operations": 0},
    "initial_metrics": initial_metrics,
    "clean_training": {name: result.to_dict() for name, result in clean_results.items()},
    "clean_metrics": clean_metrics,
    "onpolicy_training": {name: result.to_dict() for name, result in onpolicy_results.items()},
    "final_metrics": final_metrics, "constraint_checks": constraint_checks,
    "generation": generation_results, "paired_primary_contrasts": paired,
    "operations": operations, "decisions": decisions,
    "interpretation_rule": (
        "Attribute a quality benefit to U28 only if it beats the equal-budget "
        "unconstrained and matched-random controls. Claim a lower useful rank only "
        "if constrained rank64 retains at least 98% of frozen rank96 accuracy."
    ),
}
summary_path = pathlib.Path(OUTPUT_ROOT) / "summary.json"
temporary = pathlib.Path(str(summary_path) + ".tmp")
temporary.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
temporary.replace(summary_path)
torch.save({
    "contract": "frozen_u28_learned_residual_global_head_v1",
    "code_commit": CODE_COMMIT, "ranks": RANKS,
    "fixed_basis": u28_subspace.basis.cpu(), "centre": u_head.centre.detach().cpu(),
    "training_state_dict": {name: value.detach().cpu() for name, value in u_head.state_dict().items()},
    "materialized": {str(rank): snapshots[f"u28_onpolicy_r{rank}"].state_dict()
                     for rank in RANKS},
}, pathlib.Path(OUTPUT_ROOT) / "u28_constrained_global_head.pt")

import matplotlib.pyplot as plt
labels = ["dense", "frozen r96", "continued r96", "random constraint r96",
          "U28 initial r96", "U28 clean r96", "U28 on-policy r96", "U28 on-policy r64"]
names = ["dense_full", "frozen_global_r96", "continued_onpolicy_r96",
         "random28_onpolicy_r96", "u28_initial_r96", "u28_clean_r96",
         "u28_onpolicy_r96", "u28_onpolicy_r64"]
values = [accuracy(name) for name in names]
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(labels[::-1], values[::-1], color=["#2c7fb8" if "U28" in x else "#7fcdbb" for x in labels[::-1]])
ax.set_xlabel("GSM8K numeric exact match"); ax.set_title("Frozen U28 + learned trajectory residual")
ax.set_xlim(0, max(values) * 1.12); fig.tight_layout()
figure_path = pathlib.Path(OUTPUT_ROOT) / "constrained_u28_summary.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight"); plt.show()
print(json.dumps({"decisions": decisions, "paired": paired, "operations": operations}, indent=2))
print("saved:", summary_path)

## Interpretation

- If U28 rank 96 beats both equal-budget controls, freezing the discovered mechanism
  improves quality rather than merely documenting it.
- If U28 rank 96 matches but does not beat the unconstrained head, the constraint is
  quality-neutral: useful for interpretability, but not yet an optimization.
- If the random constraint performs equally well, the benefit is generic structured
  regularization rather than specific to the 28 answer directions.
- If U28 rank 64 retains at least 98% of frozen rank-96 accuracy, the constraint has
  reduced the useful global rank. That would lower head MACs from about 4.90M to
  3.27M per token. End-to-end speed must still be measured separately because CODI's
  transformer and six latent passes dominate latency.